In [2]:
"""
Figure 3: GLORYS12 Temperature and Salinity Multi-Panel Analysis
=================================================================

Custom figure showing:
- Panels a & b: Multi-depth timeseries with split trends (T & S)
- Panels c & d: Surface layer (0-50m) seasonal climatologies (T & S)
  → Saved as a separate figure (figure_03_Greenland_Sea_T_and_S_climatology.png) for potential
    supplementary materials

Features:
- Three depth ranges: 0-50m, 50-250m, 250-1000m
- Split linear trends (2015 breakpoint) with confidence intervals
- Yue & Wang (2004) modified Mann-Kendall significance testing
- Decadal seasonal climatologies
- Publication-quality formatting

Dataset: GLORYS12 Ocean Reanalysis
Region: Greenland Sea
Version: 1.3.0
Last Modified: 22-06-2026
Author: Chris Barrell
"""

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.dates as mdates
import numpy as np
import xarray as xr
import pandas as pd
import pymannkendall as mk
from pathlib import Path
from scipy import stats
from datetime import datetime
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning)

# Setup logging
import sys
sys.path.append('..')
from utils.logger import (setup_logger, log_data_loading, log_processing_step,
                          log_output_file, log_completion, log_error)

logger = setup_logger('figure_03', config_path='../config.yaml')
start_time = datetime.now()

try:
    # =============================================================================
    # CONFIGURATION
    # =============================================================================

    logger.info("Loading configuration...")

    # --- Data Configuration ---
    DATA_DIR = Path('../../glorys12_with_density/')
    DATA_PATTERN = 'glorys12_*_Greenland_Sea_with_density.nc'
    OUTPUT_DIR = Path('./outputs/figures/')
    PROCESSED_DIR = Path('./outputs/processed_data/figure_03')
    METHODS_DIR = Path('./outputs/methods')

    # --- Cache Configuration ---
    # Set True to skip data processing and load trend CSVs from disk.
    # Set False (or delete CSVs) to reprocess from raw GLORYS12 files.
    REPROCESS = False
    OUTPUT_FILE = 'figure_03_Greenland_Sea_T_and_S.png'
    OUTPUT_FILE_CLIM = 'figure_03_Greenland_Sea_T_and_S_climatology.png'

    # Create output directories
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    METHODS_DIR.mkdir(parents=True, exist_ok=True)

    # --- Depth Ranges ---
    DEPTH_RANGES = [
        {'min': 0, 'max': 50, 'label': '0-50 m'},
        {'min': 50, 'max': 250, 'label': '50-250 m'},
        {'min': 250, 'max': 1000, 'label': '250-1000 m'}
    ]

    # --- Colors for depth layers ---
    DEPTH_COLORS = {
        '0-50 m': '#66BB6A',      # Bright green - surface
        '50-250 m': '#00ACC1',    # Bright cyan/teal - intermediate
        '250-1000 m': '#1A237E',  # Very dark blue - deep
    }

    # --- Figure Configuration ---
    FIGURE_SIZE_TIMESERIES = (8, 9)   # 2-row x 1-col for panels a & b
    FIGURE_SIZE_CLIM = (8, 9)         # 2-row x 1-col for panels c & d
    DPI = 600
    FONT_SIZE_TITLE = 11
    FONT_SIZE_LABEL = 11
    FONT_SIZE_LEGEND = 10

    # --- Trend Configuration ---
    SPLIT_YEAR = 2015
    CONFIDENCE_LEVEL = 0.95
    UNCERTAINTY_METHOD = 'bootstrap'
    N_BOOTSTRAP = 1000
    ALPHA_UNCERTAINTY = 0.15
    ROLLING_WINDOW = 12

    # --- Rolling Mean Configuration ---
    SHOW_ROLLING_MEAN = True     # Set False to omit smooth overlay
    ROLLING_HALF = ROLLING_WINDOW // 2   # 6 months — excluded at each end

    # --- Monthly Data Configuration ---
    # Control which depth layers show raw monthly data beneath the rolling mean.
    # Remove a label from the set to hide monthly data for that layer.
    SHOW_MONTHLY_DATA_THETAO = {
        #'0-50 m',
        #'50-250 m',
        #'250-1000 m',
    }
    SHOW_MONTHLY_DATA_SO = {
        '0-50 m',
        '50-250 m',
        '250-1000 m',
    }

    # --- Decadal Colors ---
    DECADE_COLORS = {
        1990: '#D84315',  # Deep orange-red (earliest)
        2000: '#F57C00',  # Bright orange
        2010: '#FBC02D',  # Yellow-gold
        2020: '#7CB342',  # Yellow-green (latest complete decade)
    }

    COLOR_2025 = '#1976D2'       # Deep blue (current year)
    COLOR_CLIM_MEAN = '#424242'  # Dark gray

    logger.info(f"Split year: {SPLIT_YEAR}")
    logger.info(f"Bootstrap iterations: {N_BOOTSTRAP}")
    logger.info(f"Depth ranges: {[d['label'] for d in DEPTH_RANGES]}")

    # =============================================================================
    # STATISTICAL FUNCTIONS
    # =============================================================================

    def mann_kendall_test(x, y):
        """Yue & Wang (2004) modified Mann-Kendall test accounting for autocorrelation.

        Uses pymannkendall.yue_wang_modification_test, which corrects the variance
        of the MK statistic using serial correlation coefficients at all lags.
        Only the significance test is affected; trend magnitude is unchanged.

        References
        ----------
        Yue, S., & Wang, C. (2004). The Mann-Kendall test modified by effective
        sample size to detect trend in serially correlated hydrological series.
        Water Resources Management, 18(3), 201-218.
        https://doi.org/10.1023/B:WARM.0000043140.61082.60
        """
        y = np.asarray(y)
        y = y[~np.isnan(y)]

        result = mk.yue_wang_modification_test(y)

        if result.p < 0.001:
            significance = '***'
        elif result.p < 0.01:
            significance = '**'
        elif result.p < 0.05:
            significance = '*'
        else:
            significance = ''

        return {
            'tau': result.Tau,
            'p_value': result.p,
            'significance': significance,
            'z_statistic': result.z
        }


    def bootstrap_trend_ci(x, y, n_bootstrap=1000, confidence=0.95):
        """Bootstrap confidence intervals"""
        x = np.asarray(x)
        y = np.asarray(y)

        valid_mask = ~np.isnan(y)
        x = x[valid_mask]
        y = y[valid_mask]

        n = len(y)
        predictions = np.zeros((n_bootstrap, n))

        for i in range(n_bootstrap):
            indices = np.random.choice(n, size=n, replace=True)
            z = np.polyfit(x[indices], y[indices], 1)
            p = np.poly1d(z)
            predictions[i, :] = p(x)

        alpha = 1 - confidence
        lower_percentile = (alpha / 2) * 100
        upper_percentile = (1 - alpha / 2) * 100

        lower_bound = np.percentile(predictions, lower_percentile, axis=0)
        upper_bound = np.percentile(predictions, upper_percentile, axis=0)

        return {'lower_bound': lower_bound, 'upper_bound': upper_bound}


    def analytical_trend_ci(x, y, confidence=0.95):
        """Analytical confidence intervals"""
        x = np.asarray(x)
        y = np.asarray(y)

        valid_mask = ~np.isnan(y)
        x = x[valid_mask]
        y = y[valid_mask]

        n = len(y)
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        y_fit = slope * x + intercept

        residuals = y - y_fit
        mse = np.sum(residuals**2) / (n - 2)

        x_mean = np.mean(x)
        se_line = np.sqrt(mse * (1/n + (x - x_mean)**2 / np.sum((x - x_mean)**2)))

        t_val = stats.t.ppf((1 + confidence) / 2, n - 2)

        margin = t_val * se_line
        lower_bound = y_fit - margin
        upper_bound = y_fit + margin

        return {'lower_bound': lower_bound, 'upper_bound': upper_bound}


    def calculate_split_trends(data, time_coord, split_year):
        """Calculate split trends with CI"""
        years = time_coord.dt.year + time_coord.dt.month / 12.0
        years_numeric = years.values
        data_values = data.values

        mask_period1 = time_coord.dt.year < split_year  # TODO (future revision): update display labels 1993-2015→1993-2014 across all figures
        mask_period2 = time_coord.dt.year >= split_year

        results = {}

        # Period 1
        if np.any(mask_period1):
            years1 = years_numeric[mask_period1]
            data1 = data_values[mask_period1]

            valid_mask = ~np.isnan(data1)
            years1_valid = years1[valid_mask]
            data1_valid = data1[valid_mask]

            if len(years1_valid) > 1:
                slope1, intercept1, r_value, p_value, std_err = stats.linregress(years1_valid, data1_valid)
                trend1 = slope1 * years1 + intercept1
                slope_per_decade1 = slope1 * 10
                mk1 = mann_kendall_test(years1_valid, data1_valid)

                if UNCERTAINTY_METHOD == 'bootstrap':
                    ci_result = bootstrap_trend_ci(years1_valid, data1_valid,
                                                   n_bootstrap=N_BOOTSTRAP,
                                                   confidence=CONFIDENCE_LEVEL)
                else:
                    ci_result = analytical_trend_ci(years1_valid, data1_valid,
                                                    confidence=CONFIDENCE_LEVEL)

                ci_lower1 = np.full(len(years1), np.nan)
                ci_upper1 = np.full(len(years1), np.nan)
                ci_lower1[valid_mask] = ci_result['lower_bound']
                ci_upper1[valid_mask] = ci_result['upper_bound']

                year_start = int(years1.min())
                year_end = split_year

                results['period1'] = {
                    'time': time_coord.values[mask_period1],
                    'data': data1,
                    'trend': trend1,
                    'slope_per_decade': slope_per_decade1,
                    'mk_results': mk1,
                    'year_start': year_start,
                    'year_end': year_end,
                    'ci_lower': ci_lower1,
                    'ci_upper': ci_upper1
                }

        # Period 2
        if np.any(mask_period2):
            years2 = years_numeric[mask_period2]
            data2 = data_values[mask_period2]

            valid_mask = ~np.isnan(data2)
            years2_valid = years2[valid_mask]
            data2_valid = data2[valid_mask]

            if len(years2_valid) > 1:
                slope2, intercept2, r_value, p_value, std_err = stats.linregress(years2_valid, data2_valid)
                trend2 = slope2 * years2 + intercept2
                slope_per_decade2 = slope2 * 10
                mk2 = mann_kendall_test(years2_valid, data2_valid)

                if UNCERTAINTY_METHOD == 'bootstrap':
                    ci_result = bootstrap_trend_ci(years2_valid, data2_valid,
                                                   n_bootstrap=N_BOOTSTRAP,
                                                   confidence=CONFIDENCE_LEVEL)
                else:
                    ci_result = analytical_trend_ci(years2_valid, data2_valid,
                                                    confidence=CONFIDENCE_LEVEL)

                ci_lower2 = np.full(len(years2), np.nan)
                ci_upper2 = np.full(len(years2), np.nan)
                ci_lower2[valid_mask] = ci_result['lower_bound']
                ci_upper2[valid_mask] = ci_result['upper_bound']

                year_start = split_year
                year_end = int(years2.max())

                results['period2'] = {
                    'time': time_coord.values[mask_period2],
                    'data': data2,
                    'trend': trend2,
                    'slope_per_decade': slope_per_decade2,
                    'mk_results': mk2,
                    'year_start': year_start,
                    'year_end': year_end,
                    'ci_lower': ci_lower2,
                    'ci_upper': ci_upper2
                }

        return results


    def calculate_decadal_means(data, time_coord):
        """Calculate decadal mean climatologies"""
        years = np.unique(time_coord.dt.year.values)
        decadal_data = {}

        for decade_start in [1990, 2000, 2010]:
            decade_years = years[(years >= decade_start) & (years < decade_start + 10)]
            if len(decade_years) > 0:
                decade_mask = time_coord.dt.year.isin(decade_years)
                decade_subset = data.sel(time=decade_mask)
                monthly_clim = decade_subset.groupby('time.month').mean('time')
                monthly_clim = monthly_clim.reindex(month=np.arange(1, 13))
                decadal_data[decade_start] = monthly_clim

        decade_2020s = years[(years >= 2020) & (years < 2025)]
        if len(decade_2020s) > 0:
            decade_mask = time_coord.dt.year.isin(decade_2020s)
            decade_subset = data.sel(time=decade_mask)
            monthly_clim = decade_subset.groupby('time.month').mean('time')
            monthly_clim = monthly_clim.reindex(month=np.arange(1, 13))
            decadal_data[2020] = monthly_clim

        if 2025 in years:
            data_2025 = data.sel(time=time_coord.dt.year == 2025)
            monthly_2025 = data_2025.groupby('time.month').mean('time')
            monthly_2025 = monthly_2025.reindex(month=np.arange(1, 13))
            decadal_data[2025] = monthly_2025

        return decadal_data


    def create_decade_label(decade_start, years):
        """Create decade label"""
        if decade_start == 2025:
            return "2025"

        decade_years = years[(years >= decade_start) & (years < decade_start + 10)]

        if decade_start == 2020:
            decade_years = years[(years >= 2020) & (years < 2025)]
            if len(decade_years) > 0:
                return f"2020s ({decade_years.min()}-{decade_years.max()})"

        if len(decade_years) > 0:
            return f"{decade_start}s ({decade_years.min()}-{decade_years.max()})"

        return f"{decade_start}s"


    # =============================================================================
    # DATA PREPARATION
    # =============================================================================

    def prepare_depth_averaged_data(ds, var_name, depth_range):
        """Prepare depth-averaged data"""
        data = ds[var_name]

        data_depth = data.sel(depth=slice(depth_range['min'], depth_range['max'])).mean(dim='depth')
        data_agg = data_depth.mean(dim=['latitude', 'longitude'])
        data_agg = data_agg.compute()

        return data_agg


    def compute_rolling_mean(time_pd, data_arr):
        """
        Compute 12-month centred rolling mean with boundary masking.

        The first and last ROLLING_HALF months are set to NaN so the smooth
        line is not plotted where the centred window is underpopulated,
        avoiding spurious edge curvature.
        """
        s = pd.Series(data_arr, index=pd.to_datetime(time_pd))
        smooth = s.rolling(ROLLING_WINDOW, center=True, min_periods=6).mean().values
        smooth[:ROLLING_HALF] = np.nan
        smooth[-ROLLING_HALF:] = np.nan
        return smooth

    logger.info("Functions defined")

    # =============================================================================
    # MAIN PROCESSING
    # =============================================================================

    log_processing_step(logger, "FIGURE 3: GLORYS12 Temperature & Salinity Multi-Panel Analysis")

    # Load data
    log_data_loading(logger, 'GLORYS12', str(DATA_DIR / DATA_PATTERN))
    ds = xr.open_mfdataset(str(DATA_DIR / DATA_PATTERN), combine='by_coords', parallel=False)
    logger.info(f"Time range: {ds.time.min().dt.year.values} to {ds.time.max().dt.year.values}")

    trend_stats = {}
    climatology_data = {}

    # ==========================================================================
    # FIGURE 3: PANELS A & B — TIMESERIES (2 rows x 1 col)
    # ==========================================================================

    log_processing_step(logger, "Creating figure 3: timeseries panels (2 rows x 1 col)")

    fig_ts = plt.figure(figsize=FIGURE_SIZE_TIMESERIES)
    gs_ts = fig_ts.add_gridspec(2, 1, hspace=0.15,
                                left=0.09, right=0.98, top=0.96, bottom=0.06)

    axes_ts = [
        fig_ts.add_subplot(gs_ts[0]),  # a) Temperature timeseries
        fig_ts.add_subplot(gs_ts[1]),  # b) Salinity timeseries
    ]

    for panel_idx, var_name, var_label, units in [
        (0, 'thetao', 'Potential Temperature', '°C'),
        (1, 'so', 'Salinity', 'psu')
    ]:
        ax = axes_ts[panel_idx]
        panel_letter = chr(97 + panel_idx)  # 'a', 'b'

        logger.info(f"Panel {panel_letter}) Processing {var_label} timeseries...")

        legend_handles = []
        var_trend_stats = {}

        for depth_range in DEPTH_RANGES:
            depth_label = depth_range['label']
            color = DEPTH_COLORS[depth_label]

            logger.info(f"  Depth range: {depth_label}")

            data = prepare_depth_averaged_data(ds, var_name, depth_range)

            # Trend and MK on monthly data (consistent with other figures)
            trends = calculate_split_trends(data, ds.time, SPLIT_YEAR)
            var_trend_stats[depth_label] = trends

            if 'period1' in trends:
                p1 = trends['period1']
                logger.info(f"    Period 1 ({p1['year_start']}-{p1['year_end']}): "
                          f"{p1['slope_per_decade']:+.4f} {units}/decade "
                          f"(τ={p1['mk_results']['tau']:.3f}, p={p1['mk_results']['p_value']:.4f})")

            if 'period2' in trends:
                p2 = trends['period2']
                logger.info(f"    Period 2 ({p2['year_start']}-{p2['year_end']}): "
                          f"{p2['slope_per_decade']:+.4f} {units}/decade "
                          f"(τ={p2['mk_results']['tau']:.3f}, p={p2['mk_results']['p_value']:.4f})")

            # Plot monthly data (thin, faded)
            time_vals = ds.time.values
            show_monthly = (SHOW_MONTHLY_DATA_THETAO if var_name == 'thetao'
                            else SHOW_MONTHLY_DATA_SO)
            if depth_label in show_monthly:
                ax.plot(time_vals, data.values,
                        color=color, linewidth=1.0, alpha=0.4, zorder=2)

            # Plot 12-month rolling mean overlay
            if SHOW_ROLLING_MEAN:
                smooth = compute_rolling_mean(time_vals, data.values)
                ax.plot(time_vals, smooth,
                        color=color, linewidth=2.0, alpha=0.9, zorder=3)

            if 'period1' in trends:
                p1 = trends['period1']
                ax.fill_between(p1['time'], p1['ci_lower'], p1['ci_upper'],
                                color=color, alpha=ALPHA_UNCERTAINTY, zorder=1)
                ax.plot(p1['time'], p1['trend'],
                        color=color, linestyle='--', linewidth=2.0, alpha=0.9, zorder=4)

            if 'period2' in trends:
                p2 = trends['period2']
                ax.fill_between(p2['time'], p2['ci_lower'], p2['ci_upper'],
                                color=color, alpha=ALPHA_UNCERTAINTY, zorder=1)
                ax.plot(p2['time'], p2['trend'],
                        color=color, linestyle='--', linewidth=2.0, alpha=0.9, zorder=4)

            if 'period1' in trends and 'period2' in trends:
                label_text = f"{depth_label}\n"
                label_text += f"{p1['year_start']}-{p1['year_end']}: "
                label_text += f"{p1['slope_per_decade']:+.3f} {units} dec⁻¹ {p1['mk_results']['significance']}\n"
                label_text += f"{p2['year_start']}-{p2['year_end']}: "
                label_text += f"{p2['slope_per_decade']:+.3f} {units} dec⁻¹ {p2['mk_results']['significance']}"

                legend_handles.append(mlines.Line2D(
                    [], [], color=color, linewidth=2.5, label=label_text
                ))

        trend_stats[var_name] = var_trend_stats

        split_date = np.datetime64(f'{SPLIT_YEAR}-01-01')
        ax.axvline(x=split_date, color='#263238', linestyle=':',
                   linewidth=1.5, alpha=0.7, zorder=0)

        ax.set_title(f'{panel_letter}) Greenland Sea {var_label} Timeseries',
                     loc='left', fontsize=FONT_SIZE_TITLE)
        ax.set_ylabel(f'{var_label.lower()} ({units})', fontsize=FONT_SIZE_TITLE)

        if var_name == 'thetao':
            ax.set_ylim(top=2.75)
        elif var_name == 'so':
            ax.set_ylim(top=35.5)

        ax.set_xticks([np.datetime64(f'{y}-01-01') for y in range(1995, 2026, 5)])
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

        # NOTE: Trend statistics are encoded in legend labels rather than
        # in-panel stats boxes. Consider adding monospace stats boxes
        # (font size 9) in a future revision to align with Figures 4 & 5.
        ax.legend(handles=legend_handles, loc='upper left', ncol=2,
                  fontsize=FONT_SIZE_LEGEND-1, frameon=True, edgecolor='none', framealpha=0.25)
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

    output_path_ts = OUTPUT_DIR / OUTPUT_FILE
    fig_ts.savefig(output_path_ts, dpi=DPI, bbox_inches='tight')
    logger.info(f"Figure 3 (timeseries) saved: {output_path_ts}")
    log_output_file(logger, 'figure', output_path_ts)
    plt.close(fig_ts)

    # ==========================================================================
    # FIGURE 3 CLIMATOLOGY: PANELS C & D — SEASONAL CLIMATOLOGY (2 rows x 1 col)
    # ==========================================================================

    log_processing_step(logger, "Creating figure 3 climatology: seasonal panels (2 rows x 1 col)")

    fig_clim = plt.figure(figsize=FIGURE_SIZE_CLIM)
    gs_clim = fig_clim.add_gridspec(2, 1, hspace=0.25,
                                    left=0.09, right=0.98, top=0.96, bottom=0.06)

    axes_clim = [
        fig_clim.add_subplot(gs_clim[0]),  # c) Temperature climatology
        fig_clim.add_subplot(gs_clim[1]),  # d) Salinity climatology
    ]

    surface_depth = DEPTH_RANGES[0]  # 0-50m
    years = np.unique(ds.time.dt.year.values)
    months = np.arange(1, 13)

    for panel_idx, var_name, var_label, units in [
        (0, 'thetao', 'Potential Temperature', '°C'),
        (1, 'so', 'Salinity', 'psu')
    ]:
        ax = axes_clim[panel_idx]
        panel_letter = chr(99 + panel_idx)  # 'c', 'd'

        logger.info(f"Panel {panel_letter}) Processing {var_label} climatology (0-50 m)...")

        data = prepare_depth_averaged_data(ds, var_name, surface_depth)
        decadal_means = calculate_decadal_means(data, ds.time)
        climatology_data[var_name] = decadal_means

        for decade in sorted(decadal_means.keys()):
            mean_val = float(decadal_means[decade].mean())
            logger.info(f"  {decade}: mean = {mean_val:.3f} {units}")

        legend_handles = []

        for decade_start in sorted([k for k in decadal_means.keys() if k != 2025]):
            monthly_clim = decadal_means[decade_start]
            color = DECADE_COLORS.get(decade_start, '#333333')

            ax.plot(months, monthly_clim,
                    color=color, linewidth=2.5, alpha=0.8,
                    label=create_decade_label(decade_start, years), zorder=2)

            legend_handles.append(mlines.Line2D(
                [], [], color=color, linewidth=2.5,
                label=create_decade_label(decade_start, years)
            ))

        if 2025 in decadal_means:
            monthly_2025 = decadal_means[2025]
            ax.plot(months, monthly_2025,
                    color=COLOR_2025, linewidth=3.5, alpha=1.0, label='2025', zorder=4)
            legend_handles.append(mlines.Line2D(
                [], [], color=COLOR_2025, linewidth=3.5, label='2025'
            ))

        clim_mean = data.groupby('time.month').mean('time')
        ax.plot(months, clim_mean,
                color=COLOR_CLIM_MEAN, linewidth=3.5, linestyle='--',
                label='Climatological Mean', zorder=3)
        legend_handles.append(mlines.Line2D(
            [], [], color=COLOR_CLIM_MEAN, linewidth=3.5, linestyle='--',
            label='Climatological Mean'
        ))

        ax.set_title(f'{panel_letter}) Greenland Sea {var_label} (0-50 m) Monthly Climatology',
                     loc='left', fontsize=FONT_SIZE_TITLE)
        ax.set_ylabel(f'{var_label.lower()} ({units})', fontsize=FONT_SIZE_LABEL)
        ax.set_xlabel('Month', fontsize=FONT_SIZE_LABEL)
        ax.set_xticks(months)
        ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
        ax.legend(handles=legend_handles, loc='upper right',
                  fontsize=FONT_SIZE_LEGEND-1, frameon=True, ncol=1)
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

    output_path_clim = OUTPUT_DIR / OUTPUT_FILE_CLIM
    fig_clim.savefig(output_path_clim, dpi=DPI, bbox_inches='tight')
    logger.info(f"Figure 3 climatology saved: {output_path_clim}")
    log_output_file(logger, 'figure', output_path_clim)
    plt.close(fig_clim)

    # ==========================================================================
    # SAVE PROCESSED DATA
    # ==========================================================================

    log_processing_step(logger, "Saving processed data")

    if not REPROCESS:

        for var_name in ['thetao', 'so']:
            var_label = 'temperature' if var_name == 'thetao' else 'salinity'

            for depth_label, trends in trend_stats[var_name].items():
                depth_clean = depth_label.replace(' ', '_').replace('-', '_')

                trend_data = []

                if 'period1' in trends:
                    p1 = trends['period1']
                    trend_data.append({
                        'period': 'period1',
                        'year_start': p1['year_start'],
                        'year_end': p1['year_end'],
                        'slope_per_decade': p1['slope_per_decade'],
                        'kendall_tau': p1['mk_results']['tau'],
                        'kendall_p': p1['mk_results']['p_value'],
                        'significance': p1['mk_results']['significance']
                    })

                if 'period2' in trends:
                    p2 = trends['period2']
                    trend_data.append({
                        'period': 'period2',
                        'year_start': p2['year_start'],
                        'year_end': p2['year_end'],
                        'slope_per_decade': p2['slope_per_decade'],
                        'kendall_tau': p2['mk_results']['tau'],
                        'kendall_p': p2['mk_results']['p_value'],
                        'significance': p2['mk_results']['significance']
                    })

                df_trends = pd.DataFrame(trend_data)
                trends_file = PROCESSED_DIR / f'{var_label}_trends_{depth_clean}.csv'
                df_trends.to_csv(trends_file, index=False)
                log_output_file(logger, 'processed_data', trends_file)

        for var_name in ['thetao', 'so']:
            var_label = 'temperature' if var_name == 'thetao' else 'salinity'

            clim_data = {'month': np.arange(1, 13)}

            for decade, monthly_vals in climatology_data[var_name].items():
                clim_data[str(decade)] = monthly_vals.values

            df_clim = pd.DataFrame(clim_data)
            clim_file = PROCESSED_DIR / f'{var_label}_climatology_0_50m.csv'
            df_clim.to_csv(clim_file, index=False)
            log_output_file(logger, 'processed_data', clim_file)

    # GENERATE METHODS DOCUMENTATION
    # ==========================================================================

    methods_file = METHODS_DIR / 'figure_03_methods.md'
    with open(methods_file, 'w') as f:
        f.write("""# Figure 3: Methodology

## Temperature and Salinity Multi-Panel Analysis

### Data Sources

**GLORYS12V1 Ocean Reanalysis** (Lellouche et al., 2021)
- **Variables**: Potential temperature (thetao), Practical salinity (so)
- **Temporal Coverage**: 1993-2025
- **Spatial Resolution**: 1/12° (~8 km)
- **Vertical Resolution**: 50 depth levels (0-5500m)
- **Region**: Greenland Sea (bounded by custom polygon)

### Depth Ranges

Three depth layers analyzed:
- **Surface**: 0-50 m
- **Intermediate**: 50-250 m
- **Deep**: 250-1000 m

Depth averaging performed by selecting depth range and computing mean over depth dimension, then spatial averaging over latitude/longitude.

### Figure 3 Panels A & B: Multi-Depth Timeseries with Split Trends

#### Data Processing
1. **Depth averaging**: Mean over specified depth range
2. **Spatial averaging**: Mean over Greenland Sea region
3. **Temporal smoothing**: 12-month centered rolling mean to reduce high-frequency variability

#### Split-Trend Analysis

**Breakpoint**: 2015 (pre-specified on physical grounds)

**Periods**:
- Period 1: 1993-2015 (display label; data mask is year < 2015, i.e. up to Dec 2014)
- Period 2: 2015-2025

For each depth layer and period:

**Linear regression**:
```
y = β₀ + β₁x
```
where y is the variable value, x is decimal year (year + month/12)

**Trend expressed as**: Change per decade = 10 × β₁

#### Statistical Testing

**Modified Mann-Kendall Test** (Yue & Wang, 2004)
- Non-parametric test for monotonic trend with autocorrelation correction
- Variance of the MK statistic corrected using serial correlation coefficients at all lags
- Accounts for positive autocorrelation in monthly timeseries, which would otherwise
  inflate the probability of detecting spurious trends
- Kendall's tau (τ) measures strength and direction of trend
- Significance levels: *** (p < 0.001), ** (p < 0.01), * (p < 0.05)

**Bootstrap Confidence Intervals** (Efron & Tibshirani, 1993)
- 1000 bootstrap resamples with replacement
- 95% confidence intervals from 2.5th and 97.5th percentiles
- Displayed as shaded regions around trend lines

### Figure 3 Climatology Panels C & D: Surface Layer Seasonal Climatology

#### Depth Layer
Analysis restricted to surface layer (0-50 m) to capture seasonal cycle in upper ocean.

#### Decadal Climatologies

Mean seasonal cycles calculated for four periods:
- **1990s**: 1993-1999 (partial decade, GLORYS12 starts 1993)
- **2000s**: 2000-2009
- **2010s**: 2010-2019
- **2020s**: 2020-2024

**2025**: Shown separately (incomplete year)

**Climatological Mean**: Full 1993-2025 average

#### Calculation Method

For each decade:
1. Extract all data within decade
2. Group by month (1-12)
3. Calculate mean across all years in decade
4. Result: 12 monthly mean values

This preserves seasonal structure while showing decadal evolution.

### Physical Interpretation

#### Temperature Trends
- **Surface warming** (0-50m): Indicates atmospheric warming, reduced sea ice cover
- **Intermediate warming** (50-250m): Atlantic Water influence (Atlantification)
- **Deep layer**: Relatively stable, insulated from surface forcing

#### Salinity Trends
- **Surface freshening**: Increased ice melt, reduced brine rejection, precipitation changes
- **Intermediate layer**: Balance between Atlantic inflow and local modification
- **Deep layer**: Long-term water mass changes

#### Seasonal Climatology
- **Winter maximum T/S**: Convective mixing, Atlantic Water inflow
- **Summer minimum T/S**: Ice melt, solar heating, freshwater stratification
- **Decadal shifts**: Changing balance of ice formation, melt, and ocean heat transport

## References

Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall/CRC.

Kendall, M. G. (1975). *Rank Correlation Methods* (4th ed.). Charles Griffin.

Lellouche, J.-M., et al. (2021). The Copernicus Global 1/12° Oceanic and Sea Ice GLORYS12 Reanalysis. *Frontiers in Earth Science*, 9, 698876. https://doi.org/10.3389/feart.2021.698876

Mann, H. B. (1945). Nonparametric tests against trend. *Econometrica*, 13(3), 245-259.

Yue, S., & Wang, C. (2004). The Mann-Kendall test modified by effective sample size to detect trend in serially correlated hydrological series. *Water Resources Management*, 18(3), 201-218. https://doi.org/10.1023/B:WARM.0000043140.61082.60

## Software

Analysis performed using:
- Python 3.10.12
- xarray for NetCDF data handling
- pymannkendall for modified Mann-Kendall test (Yue & Wang, 2004)
- SciPy for statistical tests (linear regression)
- NumPy for bootstrap resampling
- Matplotlib for visualization
""")

    logger.info(f"Methods documentation saved: {methods_file}")
    log_output_file(logger, 'methods', methods_file)

    log_completion(logger, start_time)

    logger.info("="*70)
    logger.info("FIGURE 3 COMPLETE")
    logger.info("="*70)

except Exception as e:
    log_error(logger, e, context="During Figure 3 generation")
    raise

INFO     | ======================================================================
INFO     | Starting analysis: figure_03
INFO     | Log file: outputs/logs/figure_03_20260623_112840.log
INFO     | Timestamp: 2026-06-23 11:28:40
INFO     | ======================================================================
INFO     | Loading configuration...
INFO     | Split year: 2015
INFO     | Bootstrap iterations: 1000
INFO     | Depth ranges: ['0-50 m', '50-250 m', '250-1000 m']
INFO     | Functions defined
INFO     | Processing: FIGURE 3: GLORYS12 Temperature & Salinity Multi-Panel Analysis
INFO     | Loading GLORYS12 data from: ../../glorys12_with_density/glorys12_*_Greenland_Sea_with_density.nc
INFO     | Time range: 1993 to 2025
INFO     | Processing: Creating figure 3: timeseries panels (2 rows x 1 col)
INFO     | Panel a) Processing Potential Temperature timeseries...
INFO     |   Depth range: 0-50 m
INFO     |     Period 1 (1993-2015): +0.2733 °C/decade (τ=0.106, p=0.0000)
INFO     |     